## Linear Classifier in TensorFlow 
Using Low Level API in Eager Execution mode

### Load tensorflow

In [1]:
!pip install tensorflow

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from google.colab import drive
drive.mount('/content/drive')
import tensorflow as tf

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [0]:
#Collect Data

In [0]:
import numpy as np
import pandas as pd

# fix random seed for reproducibility
np.random.seed(7)
# load pima indians dataset
iris_df = pd.read_csv("/content/drive/My Drive/Colab Notebooks/Assignment_21_July/11_Iris.csv")
prices_df = pd.read_csv("/content/drive/My Drive/Colab Notebooks/Assignment_21_July/prices.csv")

In [6]:
#Enable Eager Execution if using tensflow version < 2.0
#From tensorflow v2.0 onwards, Eager Execution will be enabled by default
tf.enable_eager_execution

<function tensorflow.python.framework.ops.enable_eager_execution>

### Check all columns in the dataset

In [7]:
prices_df.columns

Index(['date', 'symbol', 'open', 'close', 'low', 'high', 'volume'], dtype='object')

### Drop columns `date` and  `symbol`

In [0]:
prices_df = prices_df.drop(["date","symbol"],axis=1)

### Consider only first 1000 rows in the dataset for building feature set and target set
Target 'Volume' has very high values. Divide 'Volume' by 1000,000

In [0]:
X = prices_df.drop("volume", axis=1)

In [0]:
y = prices_df['volume']/1000000

In [11]:
X.head(2)

,open,close,low,high
0,123.430000,125.839996,122.309998,126.250000
1,125.239998,119.980003,119.940002,125.540001


In [12]:
y.head(2)

0    2.1636
1    2.3864
Name: volume, dtype: float64

### Divide the data into train and test sets

In [0]:
from sklearn.model_selection import train_test_split
test_size = 0.30 # taking 70:30 training and test set
seed = 7  # Random numbmer seeding for reapeatability of the code
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=seed)

#### Convert Training and Test Data to numpy float32 arrays


In [14]:
Xtrain_numpy = X_train.values.astype('float32')
Xtest_numpy = X_test.values.astype('float32')
print(Xtrain_numpy)
print(type(Xtrain_numpy))

ytrain_numpy = y_train.values.astype('float32')
ytest_numpy = y_test.values.astype('float32')


[[48.21    47.87    47.62    48.52   ]
 [22.84    24.48    22.71    24.56   ]
 [54.84    55.51001 54.84    55.77   ]
 ...
 [52.33    54.63    52.06    55.72   ]
 [83.46    85.96    82.57    86.03   ]
 [74.45    74.34    73.9     74.8    ]]
<class 'numpy.ndarray'>


### Normalize the data
You can use Normalizer from sklearn.preprocessing

## Building the Model in tensorflow

1.Define Weights and Bias, use tf.zeros to initialize weights and Bias

In [0]:
#Input features
x = tf.placeholder(shape=[None,4],dtype=tf.float32, name='x-input')

#Normalize the data
x_n = tf.nn.l2_normalize(x,1)

#Actual Prices
y_ = tf.placeholder(shape=[None],dtype=tf.float32, name='y-input')



W = tf.Variable(tf.zeros(shape=[4,1]), name="Weights")
b = tf.Variable(tf.zeros(shape=[1]),name="Bias")

y = tf.add(tf.matmul(x_n,W),b,name='output')

2.Define a function to calculate prediction

In [0]:
def prediction_model():

    print(type(Xtrain_numpy))
    y = tf.add(tf.matmul(Xtrain_numpy,W),b,name='output')
    return(y)


3.Loss (Cost) Function [Mean square error]

In [17]:
loss = tf.reduce_mean(tf.square(y-y_),name='Loss')
loss

<tf.Tensor 'Loss:0' shape=() dtype=float32>

4.Function to train the Model

1.   Record all the mathematical steps to calculate Loss
2.   Calculate Gradients of Loss w.r.t weights and bias
3.   Update Weights and Bias based on gradients and learning rate to minimize loss

In [0]:
train_op = tf.train.GradientDescentOptimizer(0.03).minimize(loss)

## Train the model for 100 epochs 
1. Observe the training loss at every iteration
2. Observe Train loss at every 5th iteration

In [0]:
#Lets start graph Execution

train_op = tf.train.GradientDescentOptimizer(0.03).minimize(loss)
sess = tf.Session()

# variables need to be initialized before we can use them
sess.run(tf.global_variables_initializer())

#how many times data need to be shown to model
training_epochs = 5

In [0]:
def loss(X_train,y_train,W,b):
  yPred = tf.add(tf.matmul(X_train,W),b)
  return tf.reduce_mean(tf.square(y_train-yPred))

In [0]:
def training(X,W,b,y,learning_value):
    with tf.GradientTape() as t:
       t.watch([W,b])
       loss1 = loss(X,y,W,b)
    dw,db=t.gradient(loss1,[W,b])
    w=W-learning_value*dw
    b=b-learning_value*db
    return w,b
  

In [25]:
for epoch in range(training_epochs):
            
    Wcalc,bcalc = training(X_train,W,b,y_train,0.3)
  
    if epoch % 5 == 0:
        print ('Training loss at step: ', epoch, ' is ', loss(X_train,y_train,Wcalc,bcalc).numpy())

SystemError: ignored

In [0]:
sess.close()

### Get the shapes and values of W and b

### Model Prediction on 1st Examples in Test Dataset

## Classification using tf.Keras

In this exercise, we will build a Deep Neural Network using tf.Keras. We will use Iris Dataset for this exercise.

### Load the given Iris data using pandas (Iris.csv)

In [286]:
iris_df.head(2)

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa


### Target set has different categories. So, Label encode them. And convert into one-hot vectors using get_dummies in pandas.

In [287]:
y = pd.get_dummies(iris_df["Species"])
y.head(2)

,Iris-setosa,Iris-versicolor,Iris-virginica
0,1,0,0
1,1,0,0


### Splitting the data into feature set and target set

In [0]:
X = iris_df.drop(["Species","Id"], axis=1)

In [290]:
X.shape

(150, 4)

In [0]:
seed = 7
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.3, random_state=seed)

###  Building Model in tf.keras

Build a Linear Classifier model  <br>
1.  Use Dense Layer  with input shape of 4 (according to the feature set) and number of outputs set to 3<br> 
2. Apply Softmax on Dense Layer outputs <br>
3. Use SGD as Optimizer
4. Use categorical_crossentropy as loss function 

In [0]:
from keras.models import Sequential
from keras.layers import Dense

#Initialize Sequential Graph (model)
model = Sequential()

#Normalize input data
model.add(Dense(3, input_dim=4, activation='relu'))

#Compile the model - add Loss and Gradient Descent optimizer
model.compile(optimizer='sgd', loss='categorical_crossentropy')


### Model Training 

In [293]:
model.fit(X_train, y_train, epochs=5)

W0721 12:34:07.556974 140280745527168 deprecation.py:323] From /usr/local/lib/python3.6/dist-packages/tensorflow/python/ops/math_grad.py:1250: add_dispatch_support.<locals>.wrapper (from tensorflow.python.ops.array_ops) is deprecated and will be removed in a future version.
Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where


Epoch 1/5
105/105 [==============================] - 0s 2ms/step - loss: 7.6724
Epoch 2/5
105/105 [==============================] - 0s 211us/step - loss: 5.5181
Epoch 3/5
105/105 [==============================] - 0s 196us/step - loss: 5.4283
Epoch 4/5
105/105 [==============================] - 0s 188us/step - loss: 5.3762
Epoch 5/5
105/105 [==============================] - 0s 220us/step - loss: 5.3365


### Model Prediction

In [294]:
model.predict(X_test)

array([[0.        , 1.887924  , 0.        ],
       [0.        , 1.786764  , 0.        ],
       [2.0691278 , 1.9128294 , 0.        ],
       [0.        , 1.8376063 , 0.        ],
       [0.        , 1.5528796 , 0.        ],
       [1.6450297 , 1.5433798 , 0.        ],
       [0.        , 2.1767812 , 0.        ],
       [0.        , 1.7914436 , 0.        ],
       [1.757015  , 1.8360718 , 0.        ],
       [0.        , 2.081366  , 0.        ],
       [0.        , 2.0970678 , 0.        ],
       [0.        , 2.0171711 , 0.        ],
       [2.2226374 , 2.0848985 , 0.        ],
       [0.        , 2.2905252 , 0.        ],
       [2.1617048 , 1.9435036 , 0.        ],
       [0.        , 1.936936  , 0.        ],
       [0.        , 1.9702    , 0.        ],
       [0.        , 2.02373   , 0.        ],
       [1.9769751 , 1.9931165 , 0.        ],
       [1.9379923 , 1.8066818 , 0.        ],
       [0.        , 1.8538084 , 0.        ],
       [0.        , 2.150778  , 0.        ],
       [0.

### Save the Model

In [0]:
model.save('my_model.h5') 

### Build and Train a Deep Neural network with 2 hidden layer  - Optional - For Practice

Does it perform better than Linear Classifier? What could be the reason for difference in performance?

In [0]:
#Initialize Sequential Graph (model)
model = Sequential()

#Normalize input data
model.add(Dense(2, input_dim=4, activation='relu'))

#Add Dense layer for prediction - Keras declares weights and bias automatically
model.add(Dense(3))

#Compile the model - add Loss and Gradient Descent optimizer
model.compile(optimizer='sgd', loss='categorical_crossentropy')

In [297]:
model.predict(X_test)

array([[-2.6838431 ,  3.8362663 ,  1.5278225 ],
       [-2.3634012 ,  3.2979054 ,  1.2362583 ],
       [-1.4639992 ,  1.7235061 ,  0.33182383],
       [-2.421086  ,  3.4184952 ,  1.3209164 ],
       [-2.3397043 ,  3.3168678 ,  1.2945623 ],
       [-1.3879263 ,  1.9905951 ,  0.7992086 ],
       [-2.8157232 ,  4.2293396 ,  1.8808689 ],
       [-2.3269756 ,  3.4213097 ,  1.4539601 ],
       [-1.3849515 ,  1.6643187 ,  0.35993442],
       [-2.6404703 ,  3.806096  ,  1.5463789 ],
       [-2.862181  ,  4.258262  ,  1.8563812 ],
       [-2.5598054 ,  3.5119224 ,  1.2574002 ],
       [-1.4081708 ,  1.5134715 ,  0.12307507],
       [-3.4105663 ,  4.814166  ,  1.8588043 ],
       [-1.4876409 ,  1.8683053 ,  0.4961222 ],
       [-2.6534653 ,  3.685715  ,  1.3649572 ],
       [-2.987358  ,  4.685707  ,  2.265337  ],
       [-2.8477392 ,  4.164749  ,  1.7491407 ],
       [-1.4487658 ,  1.6556824 ,  0.26057845],
       [-1.3784263 ,  1.80639   ,  0.5619466 ],
       [-2.3195775 ,  3.4443417 ,  1.495

In [298]:
scores = model.evaluate(X_test, y_test)
scores

45/45 [==============================] - 0s 2ms/step


4.510176965925429